# D3PM
Discrete, 즉 불연속적인 공간에서 디퓨전을 활용해 생성하는 방식입니다. 이미지 자체도 엄밀히 말하자면 discrete 하기는 한데, 역시 주 목적은 텍스트 생성이죠.

이 논문은 텍스트 생성 계에 꽤 큰 영향을 주었습니다. 기존 AR 트랜스포머 기반 방식만이 아닌, 이렇게 병렬 실행이 가능한 디퓨전 또한 텍슽르르 생성할 수 있음을 보이게 됐죠. 최신 Gemini Diffusion 모델 또한 이러한 방식으로 한번에 텍스트를 생성하는 것 같구요. 최종적으로 좀 더 안정된 생성을 위한 BD3LM의 디퓨전 단계에서 이 모델의 방식을 그대로 사용합니다.

## 핵심 아이디어
텍스트 생성에 있어, 이 논문은 BERT의 training objective인 [MASK] 토큰 예측을 극단으로 치닺게 만듭니다.

기존 노이징 스텝을 바꾸어, 일정 확률로 [MASK] 토큰으로 변환해


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class TimeStepEmbedder(nn.Module):
    def __init__(self, freq_size, hidden_size):
        super(TimeStepEmbedder, self).__init__()
        self.learnable_transformation = nn.Sequential(
            nn.Linear(freq_size, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
        )
        self.freq_size = freq_size
    def get_sinusoidal(self, t, dim, theta = 10000):
        half = dim // 2
        freqs = torch.exp(
            -torch.log(theta) * torch.arange(start = 0, end=half, device=t.device) / half
        )
        args = t[:, None] * freqs[None]
        embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)

        return embedding

    def forward(self, t):
        t_freq = self.get_sinusoidal(t, self.freq_size).to(dtype = next(self.parameters()).dtype)
        t_emb = self.learnable_transformation(t_freq)
        return t_emb

In [3]:
def get_cosine_schedule(timesteps, s):
    t = torch.arange(timesteps+1, dtype=torch.float32)
    alpha_bar = torch.cos(((t / timesteps) + s) * torch.pi / 2) ** 2
    alpha_bar = alpha_bar / alpha_bar[0]
    return alpha_bar

In [12]:
alphas = get_cosine_schedule(1000, 0.008)
torch.cumprod(alphas[:-100], dim=0) # don't cumprod

print(alphas)

tensor([1.0000e+00, 9.9996e-01, 9.9991e-01,  ..., 8.8839e-05, 1.2092e-04,
        1.5793e-04])


In [13]:
def q_sample(x_0, t, alphas_cumprod, mask_token_id):
    """alpha_bar_t 확률로 원본 토큰을 마스킹한다"""
    # t에 해당하는 alpha_cumprod (스케줄링에서 이미 사실상 구한거). (B, 1, 1, ...) 형태로 브로드캐스팅 준비
    alpha_bar_t = alphas_cumprod[t].view(-1, 1)

    # 각 위치마다 MASK로 바꿀지 결정하는 마스크 생성
    # torch.rand_like는 매번 다른 난수를 생성, 이를 원래라면 훈련 루프에서 관리해 줘야 함
    rand_mask = torch.rand_like(x_0, dtype=torch.float32) > alpha_bar_t

    mask_token = torch.full_like(x_0, fill_value=mask_token_id)
    # rand_mask가 True인 위치는 MASK 토큰으로, False인 위치는 원래 토큰 x_0으로
    x_t = torch.where(rand_mask, mask_token, x_0)

    return x_t

In [14]:
from rotary_embedding_torch import RotaryEmbedding
from einops import rearrange, einsum
from einops.layers.torch import Rearrange
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, heads):
        super().__init__()
        self.heads = heads
        self.head_dim = hidden_size // heads
        self.scale = self.head_dim ** -0.5

        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.out_proj = nn.Linear(hidden_size, hidden_size)

        self.pos_emb = RotaryEmbedding(dim=self.head_dim)
        self.split_heads = Rearrange('b n (h d) -> b h n d', h = heads)

    def forward(self, x):
        q, k, v = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        q, k, v = map(self.split_heads, (q, k, v))

        q, k = self.pos_emb.rotate_queries_or_keys(q), self.pos_emb.rotate_queries_or_keys(k)
        sim = einsum(q, k, 'b h q_len d, b h k_len d -> b h q_len k_len')
        attn = sim.softmax(dim=-1)

        ctx = einsum(attn, v, 'b h q_len k_len, b h k_len d -> b h q_len d')
        attn_out = rearrange(ctx, 'b h n d -> b n (h d)')
        return self.out_proj(attn_out)

class TransformerBlock(nn.Module):
    def __init__(self, hidden_size, heads, ffn_size):
        super().__init__()
        self.attn = MultiHeadAttention(hidden_size, heads)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, ffn_size),
            nn.GELU(),
            nn.Linear(ffn_size, hidden_size)
        )
        self.norm1 = nn.LayerNorm(hidden_size)
        self.norm2 = nn.LayerNorm(hidden_size)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

In [15]:
class D3PM(nn.Module):
    def __init__(self, vocab_size, num_layers, hidden_size, ffn_size, heads):
        super(D3PM, self).__init__()
        self.vocab_size = vocab_size
        self.tokenizer = nn.Embedding(vocab_size + 1, hidden_size) # +1 for MASK token
        self.timestep_embed = TimeStepEmbedder(hidden_size, hidden_size)

        self.transformer_blocks = nn.ModuleList(
            [TransformerBlock(hidden_size, heads, ffn_size) for _ in range(num_layers)]
        )
        self.norm_out = nn.LayerNorm(hidden_size)
        self.head = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, t):
        token_emb = self.tokenizer(x)
        time_emb = self.timestep_embed(t).unsqueeze(1)

        x_emb = token_emb + time_emb

        for block in self.transformer_blocks:
            x_emb = block(x_emb)

        x_out = self.norm_out(x_emb)
        logits = self.head(x_out)
        return logits

In [16]:
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
def train():
    # Hyperparam
    device = "cuda" if torch.cuda.is_available() else "cpu"
    timesteps = 1000
    vocab_size = 256  # 0-255 사이의 값으로 가정
    mask_token_id = vocab_size # MASK 토큰은 어휘 크기 바로 다음 인덱스

    # model setting
    num_layers = 6
    hidden_size = 256
    ffn_size = hidden_size * 4
    heads = 4
    seq_len = 256

    batch_size = 32
    epochs = 10
    lr = 1e-4

    # dummies
    dummy_data = torch.randint(0, vocab_size, (batch_size * 10, seq_len))
    dataset = TensorDataset(dummy_data)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # init models
    model = D3PM(vocab_size, num_layers, hidden_size, ffn_size, heads).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    # alpha_bar
    alphas_cumprod = get_cosine_schedule(timesteps, 0.008).to(device)
    #alphas_cumprod = torch.cumprod(alphas, dim=0) # only linear에만 활용하는 게 맞는듯

    print(f"{device} 활용중...") # 저 이제 쿠다 씁니다.

    for epoch in range(epochs):
        pbar = tqdm(dataloader)
        total_loss = 0
        for step, (x_0,) in enumerate(pbar):
            optimizer.zero_grad()

            x_0 = x_0.to(device)
            b, l = x_0.shape
            # 랜덤 타임스텝에서 x_0을 찾는 방식. 랜덤 관리 등등 간단해서 이렇게 했음. 다음과 같이 해도 됨
            # 1. 지정해둔 timesteps만큼 노이즈 추가, 2. t-1, t-2씩 돌아가며 x_0 예측, 예측한 x_hat_0에 q 작업 적용시켜 실제 x_t랑 CE
            t = torch.randint(0, timesteps, (b,), device=device).long()

            x_t = q_sample(x_0, t, alphas_cumprod, mask_token_id)

            predicted_x_0_logits = model(x_t, t)

            loss = F.cross_entropy(predicted_x_0_logits.view(-1, vocab_size), x_0.view(-1))

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            pbar.set_description(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss / (step+1):.4f}")


In [17]:
train()

cuda 활용중...


Epoch 1/10 | Loss: 5.3730: 100%|██████████| 10/10 [00:01<00:00,  9.23it/s]
Epoch 2/10 | Loss: 4.6918: 100%|██████████| 10/10 [00:00<00:00, 16.29it/s]
Epoch 3/10 | Loss: 4.0952: 100%|██████████| 10/10 [00:00<00:00, 15.98it/s]
Epoch 4/10 | Loss: 3.7778: 100%|██████████| 10/10 [00:00<00:00, 16.12it/s]
Epoch 5/10 | Loss: 3.3233: 100%|██████████| 10/10 [00:00<00:00, 16.32it/s]
Epoch 6/10 | Loss: 3.1262: 100%|██████████| 10/10 [00:00<00:00, 15.70it/s]
Epoch 7/10 | Loss: 2.8738: 100%|██████████| 10/10 [00:00<00:00, 15.82it/s]
Epoch 8/10 | Loss: 3.0261: 100%|██████████| 10/10 [00:00<00:00, 16.20it/s]
Epoch 9/10 | Loss: 2.8480: 100%|██████████| 10/10 [00:00<00:00, 15.92it/s]
Epoch 10/10 | Loss: 2.9438: 100%|██████████| 10/10 [00:00<00:00, 16.18it/s]
